## 设置参数

In [ ]:
import os
import csv

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

import scanpy as sc
import scvelo as scv

import seaborn as sns
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib import rcParams
from tqdm import tqdm

import requests
import multiprocessing
from joblib import Parallel, delayed

scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.settings.set_figure_params('')

scv.set_figure_params()
scv.set_figure_params(dpi_save=300, frameon=False, figsize=(8, 6))

plt.rc('font',family='Times New Roman') 
rcParams['mathtext.default'] = 'regular'

import argparse
parser = argparse.ArgumentParser()
parser.add_argument( '--dataset_name', type=str, default="intestinal_organoid", help='pancreas, gastrulation_erythroid, dentategyrus_lamanno, intestinal_organoid, cong_velocity') 
parser.add_argument( '--n_jobs', type=int, default=-1, help='number of cpus to use')
# parser.add_argument( '--var_names', type=str, default="all", help='all, highly_variable_genes')
parser.add_argument( '--n_neighbors', type=int, default=30, help='number of neighbors')
parser.add_argument( '--n_top_genes', type=int, default=2000, help='n_top_genes')
# parser.add_argument( '--TF_databases', nargs='+', default='ENCODE ChEA', help='knockTF ChEA ENCODE')
# parser.add_argument( '--max_n_TF', type=int, default=99, help='max number of TFs')
parser.add_argument( '--max_iter', type=int, default=1000, help='max number of iterations')
parser.add_argument( '--max_tau', type=int, default=5, help='max number of tau')
# parser.add_argument( '--n_time_points', type=int, default=1000, help='use_raw')
# parser.add_argument( '--use_raw', type=int, default=0, help='use_raw')
parser.add_argument( '--basis', type=str, default='umap', help='umap')
# args = parser.parse_args()
args = parser.parse_args(args=[])
args.result_path = args.dataset_name + '/'

## 数据预处理

In [ ]:
# adata = sc.read(args.result_path + args.dataset_name + '_pp.h5ad')
# # 提取spliced和unspliced数据到csv文件
# spliced = adata.layers['spliced'].toarray()
# unspliced = adata.layers['unspliced'].toarray()
# spliced_df = pd.DataFrame(spliced, index=adata.obs_names, columns=adata.var_names).T
# unspliced_df = pd.DataFrame(unspliced, index=adata.obs_names, columns=adata.var_names).T
# spliced_df.to_csv(args.result_path + args.dataset_name + '_spliced.csv')
# unspliced_df.to_csv(args.result_path + args.dataset_name + '_unspliced.csv')


In [ ]:
if os.path.exists(args.result_path + args.dataset_name + '_pp.h5ad'):
    print('The result file already exists!')
    adata = sc.read(args.result_path + args.dataset_name + '_pp.h5ad')
    df = pd.read_csv(args.result_path + args.dataset_name + '_pseudoTime.csv')

else:    
    if args.dataset_name == 'pancreas':
        adata = scv.datasets.pancreas()
        adata.uns['selected_genes'] = ['Actn4', 'Cpe', 'Ece1', 'Litaf', 'Maml3', 'Nnat', 'Ppp3ca']

    elif args.dataset_name == 'gastrulation_erythroid':
        adata = scv.datasets.gastrulation_erythroid()   
        adata.obs['clusters'] = adata.obs['celltype'].copy()
        adata.uns['selected_genes'] = ['Blvrb', 'Hsp90ab1', 'Mllt3', 'Skap1', 'Tacc1']

    elif args.dataset_name == 'dentategyrus_lamanno':
        adata = scv.datasets.dentategyrus_lamanno()
        adata.uns['selected_genes'] = ['Dcx']
        adata.obs_names = [i[8:-1] for i in adata.obs_names]

    elif args.dataset_name == 'intestinal_organoid' or args.dataset_name == 'cong_velocity':
        adata = sc.read('./data/Organoids/organoids.h5ad')
        cell_mapper = {
            '1': 'Enterocytes',
            '2': 'Enterocytes',
            '3': 'Enteroendocrine',
            '4': 'Enteroendocrine progenitor',
            '5': 'Tuft cells',
            '6': 'TA cells',
            '7': 'TA cells',
            '8': 'Stem cells',
            '9': 'Paneth cells',
            '10': 'Goblet cells',
            '11': 'Stem cells',
        }
        ensembl_symbol_converter_url = "https://www.biotools.fr/mouse/ensembl_symbol_converter/?presenter=none"
        ensembl_id = '\n'.join(adata.var_names)
        response = requests.post(ensembl_symbol_converter_url, data={'input_data':ensembl_id,'response_data':None},proxies={'http':'http://localhost:7890','https':'http://localhost:7890'})
        gene_symbol = response.text.split("\n")[:-1]
        adata.var_names = gene_symbol
        adata.obs['cell_type'] = adata.obs.som_cluster_id.map(cell_mapper).astype('str')
        adata.obs['clusters'] = adata.obs['cell_type'].copy()
        adata.uns['selected_genes'] = ['Atp2a3','Clu','Edn1','Gsta4','H1f0','Muc13','Ndrg1','Slc7a9']

    if not os.path.exists(args.result_path):
        os.makedirs(args.result_path)

    gene_names = []
    for tmp in adata.var_names:
        gene_names.append(tmp.capitalize())
    adata.var_names = gene_names
    adata.var_names_make_unique()
    adata.obs_names_make_unique()

    # 预处理数据，计算瞬时速度
    if args.dataset_name != 'cong_velocity' and args.dataset_name != 'intestinal_organoid':
        scv.pp.filter_and_normalize(adata, min_shared_counts=20, n_top_genes=args.n_top_genes)
    #     scv.pp.moments(adata, n_pcs=30, n_neighbors=args.n_neighbors)
    #     scv.tl.recover_dynamics(adata, n_jobs=args.n_jobs)
    #     scv.tl.velocity(adata, mode='dynamical')

    # v_intestinal = pd.read_excel(rf'E:\Administrator\Down\MobileFile\v_intestinal.xlsx',header=None)

    scv.pp.moments(adata, n_pcs=30, n_neighbors=args.n_neighbors)
    scv.tl.recover_dynamics(adata, n_jobs=args.n_jobs)
    velocity_file = 'v_cong.csv' if args.dataset_name == 'cong_velocity' else f'v_{args.dataset_name}.csv'
    v_intestinal = pd.read_csv(velocity_file, header=None)
    adata.layers['velocity'] = v_intestinal.values.T
    scv.tl.velocity_graph(adata, n_jobs=args.n_jobs)
    
    scv.tl.recover_dynamics(adata, var_names='velocity_scv_genes', n_jobs=args.n_jobs)
    scv.tl.velocity(adata, vkey='velocity_scv_dynamical', mode='dynamical')
    scv.tl.velocity_graph(adata, vkey='velocity_scv_dynamical', n_jobs=args.n_jobs)

    scv.tl.velocity(adata, vkey='velocity_scv_stochastic', mode='stochastic')
    scv.tl.velocity_graph(adata, vkey='velocity_scv_stochastic', n_jobs=args.n_jobs)

    scv.tl.velocity(adata, vkey='velocity_scv_deterministic', mode='deterministic')
    scv.tl.velocity_graph(adata, vkey='velocity_scv_deterministic', n_jobs=args.n_jobs)

    if args.dataset_name == 'dentategyrus_lamanno':
        sc.tl.umap(adata)

    # 伪时间计算
    scv.tl.latent_time(adata)
    sorted_indices = np.argsort(adata.obs['latent_time'].values)
    adata.obs['latent_time_sorted'] = adata.obs['latent_time'].values[sorted_indices]
    adata.uns['cell_sorted_dict'] = {cell: i for i, cell in enumerate(adata.obs_names[sorted_indices])}

    adata.uns['selected_genes_index'] = [adata.var_names.get_loc(gene) for gene in adata.uns['selected_genes']]

    # 创建包含伪时间和聚类信息的数据框
    df = pd.DataFrame({
        'Cell': adata.obs_names,
        'PseudoTime': adata.obs['latent_time'].values,
        'Cluster': adata.obs['clusters'].values
    })

    # 获取按照伪时间排序后的聚类名称
    adata.obs['clusters_sorted'] = pd.Categorical(
        adata.obs['clusters'], 
        categories=df.groupby('Cluster')['PseudoTime'].min().sort_values().index.values.tolist(), 
        ordered=True
    )
    adata.obs['clusters_sorted_reversed'] = pd.Categorical(
        adata.obs['clusters'],
        categories=df.groupby('Cluster')['PseudoTime'].min().sort_values(ascending=False).index.values.tolist(),
        ordered=True
    )

    df.to_csv(args.result_path + args.dataset_name + '_pseudoTime.csv', index=False)

    # 保存数据
    adata.write(args.result_path + args.dataset_name + '_pp.h5ad')

In [ ]:
gene_expr = adata[:, adata.uns['selected_genes']].X.toarray()

df = pd.DataFrame(gene_expr, columns=adata.uns['selected_genes'], index=adata.obs_names)
df['latent_time'] = adata.obs['latent_time']
df['cluster'] = adata.obs['clusters']
df_sorted = df.sort_values('latent_time')

plt.figure(figsize=(12, 8))
sns.heatmap(df_sorted[adata.uns['selected_genes']].T, cmap='viridis', cbar_kws={'label': 'Expression'})
plt.title('Gene Expression Heatmap (Sorted by Pseudotime)')
plt.xlabel('Cells')
plt.ylabel('Genes')
plt.tight_layout()
plt.savefig(args.result_path + args.dataset_name + '_gene_expression_heatmap.png', dpi=300)
plt.close()

## 绘制伪时间折线图

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(adata.obs['latent_time_sorted'].values, label='Pseudo-time')
plt.xlabel('Sorted Cell Index')
plt.ylabel('Pseudo-time')
plt.legend()
plt.savefig(args.result_path + args.dataset_name + '_latent_time.png',dpi=300)
plt.close()

## 绘制聚类在UMAP中的分布图

In [ ]:
scv.pl.scatter(adata, color='clusters', basis=args.basis, title='Cluster', dpi=300, save=args.result_path + 'cluster.png', show=False)

## 绘制伪时间

In [ ]:
scv.pl.scatter(adata, color='latent_time', basis=args.basis, color_map='gnuplot', save=args.result_path + 'latent_time.png', dpi=300, show=False)

## 绘制速度流

In [ ]:
scv.pl.velocity_embedding_stream(adata, basis=args.basis, title='Fate Trajectory', dpi=300, save=args.result_path + 'velocity_stream.png', show=False)

## 绘制所有簇的小提琴图和箱线图

In [ ]:
df = pd.read_csv(args.result_path + args.dataset_name + '_pseudoTime.csv')
# 设置调色板
palette = sns.color_palette("husl", len(adata.obs['clusters_sorted'].cat.categories))

# 将聚类标签替换为它们的索引
cluster_to_index = {cluster: index for index, cluster in enumerate(adata.obs['clusters_sorted'].cat.categories)}
df['ClusterIndex'] = df['Cluster'].map(cluster_to_index)

# 设置图片分辨率和字体大小
sns.set_context("paper", font_scale=1.5)

# 绘制伪时间在不同聚类中的分布（按最低伪时间排序）
plt.figure(figsize=(12, 8))
sns.violinplot(
    x='ClusterIndex', 
    y='PseudoTime', 
    data=df, 
    order=range(len(cluster_to_index)), 
    palette=palette, 
    inner=None
)
sns.swarmplot(x='ClusterIndex', y='PseudoTime', data=df, order=range(len(cluster_to_index)), color='k', alpha=0.5)
plt.title('Distribution of Pseudo-time Across clusters (Ordered by Minimum Pseudo-time)', fontsize=20)
plt.xlabel('Cluster Index', fontsize=16)
plt.ylabel('Pseudo-time', fontsize=16)

# 添加图例
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=10) for i in range(len(cluster_to_index))]
legend = plt.legend(handles, adata.obs['clusters_sorted'].cat.categories, title='Cluster', fontsize=12, title_fontsize=14)
plt.setp(legend.get_title(), fontweight='bold')

plt.tight_layout()
plt.savefig(args.result_path + 'violinplot.png', dpi=300)
plt.close()

# 绘制每个聚类在伪时间中的位置（按最低伪时间排序）
plt.figure(figsize=(12, 8), dpi=300)
sns.boxplot(x='ClusterIndex', y='PseudoTime', data=df, order=range(len(cluster_to_index)), palette=palette)
plt.title('Boxplot of Pseudo-time in Each cluster (Ordered by Minimum Pseudo-time)', fontsize=20)
plt.xlabel('Cluster Index', fontsize=16)
plt.ylabel('Pseudo-time', fontsize=16)

# 添加图例
handles = [plt.Line2D([0], [0], color=palette[i], lw=4) for i in range(len(cluster_to_index))]
legend = plt.legend(handles, adata.obs['clusters_sorted'].cat.categories, title='Cluster', fontsize=12, title_fontsize=14)
plt.setp(legend.get_title(), fontweight='bold')

plt.tight_layout()
plt.savefig(args.result_path + 'boxplot.png', dpi=300)
plt.close()

## 绘制聚类气泡图

In [ ]:
plt.figure(figsize=(6, 4), dpi=300)
sc.pl.dotplot(adata, var_names=adata.uns['selected_genes'], groupby='clusters_sorted_reversed', standard_scale='var', cmap='Blues', dot_min=0.1, dot_max=1, show=False)
plt.xlabel('Genes', fontsize=14)
plt.ylabel('Clusters', fontsize=14)
plt.savefig(args.result_path + 'dotplot.png', dpi=300, bbox_inches='tight')
plt.close()

# 绘制靶基因的UMAP图

In [ ]:
for gene in adata.uns['selected_genes']:
    scv.pl.umap(adata, color=gene, dpi=300, save=args.result_path + gene + '_umap.png', show=False)

## 计算不同$\alpha$下的最优$\tau$并绘制热图

In [ ]:
# 这里可能会报磁盘空间不足的错误，可以分两批处理alphas
alphas = [5,4.5,4,3.5,3,2.5,2,1.5,1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001, 0.0005, 0.0001, 0.00005, 0.00001, 0.000005, 0.000001, 0.0000005, 0.0000001, 0.00000005, 0.00000001, 0.000000005, 0.000000001, 0.0000000005, 0.0000000001,0]

# 读取伪时间和聚类信息
pseudo_cells_dict = pd.read_csv(
    args.result_path + args.dataset_name + '_pseudoTime.csv', 
    index_col='Cell', 
    dtype={'Cell': str}
)['PseudoTime'].to_dict()

def filter_gene_regulation(gene_file, selected_genes):
    selected_genes_lower = {gene.lower(): gene for gene in selected_genes}
    filtered_dict = {}
    with open(gene_file, 'r') as f:
        for line in f:
            if line.strip():
                parts = line.strip().split('\t')
                gene_name = parts[0]
                if gene_name.lower() in selected_genes_lower:
                    filtered_dict[selected_genes_lower[gene_name.lower()]] = [item.capitalize() for item in parts[2:] if item.capitalize() in adata.var_names]
    return filtered_dict
filtered_dict1 = filter_gene_regulation('ENCODE_attribute_set_library_crisp.gmt', adata.uns['selected_genes'])
filtered_dict2 = filter_gene_regulation('ChEA_attribute_set_library_crisp.gmt', adata.uns['selected_genes'])
# 合并两个字典
filtered_dict = {key: list(set(filtered_dict1.get(key, []) + filtered_dict2.get(key, []))) for key in adata.uns['selected_genes']}

if os.path.exists(args.result_path + args.dataset_name + '_best_tau_by_alpha.csv'):
    print(f'The {args.dataset_name}_best_tau_by_alpha.csv already exists!')
    df = pd.read_csv(args.result_path + args.dataset_name + '_best_tau_by_alpha.csv')
else:
    # 定义时间延迟范围
    time_delays = range(1, args.max_tau+1)

    # 处理单个基因和聚类的函数，使用 Lasso 回归
    def process_cluster_gene(adata, cluster_cells, gene_index, filtered_dict, pseudo_cells_dict, t, alpha):
        X_all = []
        y_all = []

        for cell_name in cluster_cells:
            cell_index = adata.obs_names.get_loc(cell_name)
            cell_pos = adata.uns['cell_sorted_dict'][cell_name]

            if cell_pos - t < 0:
                continue  # 跳过无效的时间延迟组合

            new_cell_name = [key for key, value in adata.uns['cell_sorted_dict'].items() if value == cell_pos - t][0]
            new_cell_index = adata.obs_names.get_loc(new_cell_name)

            X_cell = adata.layers['Ms'][new_cell_index]
            y = adata.layers['velocity_scv_dynamical'][cell_index, gene_index]

            gene_name = adata.var_names[gene_index]
            tf_genes = filtered_dict.get(gene_name, [])
            tf_indices = [adata.var_names.get_loc(tf) for tf in tf_genes if tf in adata.var_names]
            if len(tf_indices) == 0:
                continue  # 跳过没有TF的数据

            X_filtered = X_cell[tf_indices].reshape(1, -1)

            X_all.append(X_filtered)
            if not np.isnan(y):
                y_all.append(y)

        if len(X_all) == 0:
            return None  # 没有有效数据
        
        if len(y_all) == 0:
            return None  # 没有有效数据

        X_all = np.vstack(X_all)
        y_all = np.array(y_all)

        scaler = StandardScaler()
        X_all = scaler.fit_transform(X_all)

        lasso = Lasso(alpha=alpha, max_iter=10000, fit_intercept=False)
        lasso.fit(X_all, y_all)

        current_loss = np.mean((lasso.predict(X_all) - y_all) ** 2)
        return t, current_loss

    def find_best_time_delay_for_cluster(adata, cluster_cells, gene_index, filtered_dict, pseudo_cells_dict, alpha):
        best_loss = float('inf')
        best_t = None

        for t in time_delays:
            result = process_cluster_gene(adata, cluster_cells, gene_index, filtered_dict, pseudo_cells_dict, t, alpha)
            if result is None:
                continue  # 跳过无效的时间延迟组合
            t, loss = result

            if loss < best_loss:
                best_loss = loss
                best_t = t
        return best_t

    def optimize_time_delays_across_clusters(adata, filtered_dict, pseudo_cells_dict, alpha):
        clusters = adata.obs['clusters'].unique().tolist()
        results = Parallel(n_jobs=1)(delayed(find_best_time_delay_for_cluster)(adata, adata.obs[adata.obs['clusters'] == cluster].index, gene_index, filtered_dict, pseudo_cells_dict, alpha) for cluster in clusters for gene_index in adata.uns['selected_genes_index'])
        # results = [find_best_time_delay_for_cluster(adata, adata.obs[adata.obs['clusters'] == cluster].index, gene_index, filtered_dict, pseudo_cells_dict, alpha) for cluster in clusters for gene_index in adata.uns['selected_genes_index']]
        return results

    def save_results_to_csv(filename, data):
        with open(filename, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(data)

    def plot_time_delay_heatmap(cluster_best_t, alpha, selected_genes):
        clusters = adata.obs['clusters'].unique().tolist()
        df = pd.DataFrame(index=clusters, columns=selected_genes)

        for cluster in clusters:
            for gene_index in adata.uns['selected_genes_index']:
                # print(cluster, gene_index)
                df.loc[cluster, adata.var_names[gene_index]] = float(cluster_best_t[cluster][gene_index])

        df = df.apply(pd.to_numeric)
        custom_order = adata.obs['clusters_sorted_reversed'].cat.categories.tolist()
        df = df.reindex(custom_order)

        plt.figure(figsize=(10, 8), dpi=300)
        heatmap = sns.heatmap(df, annot=True, cmap='coolwarm', cbar_kws={'label': 'Time Delay'})
        plt.title(f"alpha={alpha}: Best time delay for each gene in each cluster", fontsize=14)
        plt.xlabel('Genes', fontsize=12)
        plt.ylabel('Clusters', fontsize=12)
        plt.xticks(fontsize=10)
        plt.yticks(fontsize=10)
        plt.savefig(args.result_path + f'heatmap_alpha_{alpha}.png', dpi=300)
        plt.close()  # 关闭当前图形，释放内存

    output_file = args.result_path + args.dataset_name + '_best_tau_by_alpha.csv'

    # 写入CSV文件头
    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['alpha', 'cluster', 'gene', 'best_tau'])

    for alpha in alphas:
        cluster_best_t = {}
        results = optimize_time_delays_across_clusters(adata, filtered_dict, pseudo_cells_dict, alpha)
        for result, (cluster, gene_index) in zip(results, [(cluster, gene_index) for cluster in adata.obs['clusters'].unique().tolist() for gene_index in adata.uns['selected_genes_index']]):
            best_t = result
            gene = adata.var_names[gene_index]
            save_results_to_csv(output_file, [alpha, cluster, gene, best_t])
            if cluster not in cluster_best_t:
                cluster_best_t[cluster] = {}
            cluster_best_t[cluster][gene_index] = best_t

        plot_time_delay_heatmap(cluster_best_t, alpha, adata.var_names[adata.uns['selected_genes_index']])
    df = pd.read_csv(args.result_path + args.dataset_name + '_best_tau_by_alpha.csv')


## 绘制基因随簇的最佳时间延迟变化折线图

In [ ]:
# 绘制不同alpha不同基因的随簇的最佳时间延迟变化的折线图
for alpha in alphas:
    for gene in adata.uns['selected_genes']:
        df_gene = df[(df['alpha'] == alpha) & (df['gene'] == gene)]
        df_gene['cluster'] = pd.Categorical(df_gene['cluster'], categories=adata.obs['clusters_sorted'].cat.categories.tolist(), ordered=True)
        df_gene = df_gene.sort_values('cluster')
        plt.figure(figsize=(6, 4), dpi=300)
        sns.lineplot(x='cluster', y='best_tau', data=df_gene, marker='o')
        plt.title(f'Best time delay for {gene} across clusters (alpha={alpha})', fontsize=14)
        plt.xlabel('Cluster', fontsize=12)
        plt.ylabel('Best time delay', fontsize=12)
        plt.ylim(0.5, args.max_tau+0.5)
        plt.yticks(range(1,args.max_tau+1))
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.savefig(args.result_path + f'best_tau_{gene}_alpha_{alpha}.png', dpi=300)
        plt.close()

## 使用神经网络拟合TF权重

In [ ]:
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


# if os.path.exists(args.result_path + 'predict_weights.csv'):
#     print(f'The {args.result_path}predict_weights.csv already exists!')
#     df = pd.read_csv(args.result_path + 'predict_weights.csv')
#     def safe_json_loads(value):
#         if isinstance(value, str):
#             return json.loads(value)
#         return value
#     df['Selected Genes'] = df['Selected Genes'].apply(safe_json_loads)
#     df['Selected Weights'] = df['Selected Weights'].apply(safe_json_loads)
# else:
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 读取和处理数据
# adata = sc.read(args.result_path + args.dataset_name + '_pp.h5ad')
# pseudo_cells_dict = pd.read_csv(
#     args.result_path + args.dataset_name + '_pseudoTime.csv', 
#     index_col='Cell', 
#     dtype={'Cell': str}
# )['PseudoTime'].to_dict()
# df = pd.read_csv(args.result_path + args.dataset_name + '_best_tau_by_alpha.csv')
df = df[df['alpha'] == 0].drop(columns=['alpha']).set_index(['cluster', 'gene']).unstack().droplevel(0, axis=1)
cluster_best_t = df.to_dict(orient='index')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {device}')

class SimpleNN(nn.Module):
    def __init__(self, input_size):
        super(SimpleNN, self).__init__()
        self.fc = nn.Linear(input_size, 1, bias=False)

    def forward(self, x):
        return self.fc(x)

def process_cell_gene(adata, v_key, gene, filtered_dict, cell_name, t, epoch=1000):
    cell_index = adata.obs_names.get_loc(cell_name)
    cell_pos = adata.uns['cell_sorted_dict'][cell_name]
    new_cell_pos = cell_pos - t

    if new_cell_pos < 0:
        return None
    
    new_cell_name = [key for key, value in adata.uns['cell_sorted_dict'].items() if value == new_cell_pos][0]
    new_cell_index = adata.obs_names.get_loc(new_cell_name)

    X_cell = adata.layers['Ms'][new_cell_index]
    y = adata.layers[v_key][cell_index, adata.var_names.get_loc(gene)]

    tf_genes = filtered_dict.get(gene, [])
    tf_indices = [adata.var_names.get_loc(tf) for tf in tf_genes if tf in adata.var_names]
    if len(tf_indices) == 0:
        return None

    X_filtered = X_cell[tf_indices].reshape(1, -1)
    X_filtered = (X_filtered - np.mean(X_filtered)) / np.std(X_filtered)
    X_filtered = torch.tensor(X_filtered, dtype=torch.float32, device=device, requires_grad=True)
    y = torch.tensor([y], dtype=torch.float32, device=device)

    model = SimpleNN(input_size=X_filtered.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.001, weight_decay=1e-5)

    losses = []
    model.train()
    for epoch in range(epoch):
        optimizer.zero_grad()
        output = model(X_filtered)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            model.fc.weight.data = torch.round(model.fc.weight.data * 10000000) / 10000000
        losses.append(loss.item())
    
    current_loss = loss.item()
    return model.fc.weight.grad.cpu().detach().numpy().flatten(), output.cpu().detach().numpy().flatten()[0], current_loss, losses

results = []
all_losses = []
loss_labels = []
gene_cluster_losses = {}
mid_cell_losses = []

# 计算所有靶基因在所有簇最中间的细胞上的权重和预测
for gene_one in adata.uns['selected_genes']:
    gene_cluster_losses[gene_one] = []
    for cluster in adata.obs['clusters_sorted'].cat.categories.tolist():
        cluster_cells = adata.obs[adata.obs['clusters'] == cluster].index
        cluster_cells = sorted(cluster_cells, key=lambda x: pseudo_cells_dict[x])
        if len(cluster_cells) & 1:
            cell_name = cluster_cells[len(cluster_cells) // 2 + 1]
        else:
            cell_name = cluster_cells[len(cluster_cells) // 2]
        result = process_cell_gene(adata, 'velocity_scv_dynamical', gene_one, filtered_dict, cell_name, cluster_best_t[cluster][gene_one])
        if result is None:
            continue
        weight, prediction, loss, losses = result
        gene_cluster_losses[gene_one].append((cluster, losses))
        results.append({
            'Cluster': cluster,
            'Cell': cell_name,
            'Gene': gene_one,
            'Selected Genes': filtered_dict[gene_one],
            'Selected Weights': weight
        })
        # 存入预测速度
        cell_idx = adata.obs_names.get_loc(cell_name)
        gene_idx = adata.var_names.get_loc(gene_one)

# 计算所有细胞中最中间的细胞上的权重和预测
all_cells = adata.obs.index
all_cells = sorted(all_cells, key=lambda x: pseudo_cells_dict[x])
if len(all_cells) & 1:
    cell_name = all_cells[len(all_cells) // 2 + 1]
else:
    cell_name = all_cells[len(all_cells) // 2]
cluster_one = adata.obs.loc[cell_name, 'clusters']
for gene in adata.uns['selected_genes']:
    result = process_cell_gene(adata, 'velocity_scv_dynamical', gene, filtered_dict, cell_name, cluster_best_t[cluster_one][gene])
    if result is None:
        continue
    weight, prediction, loss, losses = result
    mid_cell_losses.append((gene, losses))
    results.append({
        'Cluster': 'MID',
        'Cell': cell_name,
        'Gene': gene,
        'Selected Genes': filtered_dict[gene],
        'Selected Weights': weight
    })
    cell_idx = adata.obs_names.get_loc(cell_name)
    gene_idx = adata.var_names.get_loc(gene)

# 将结果转换为DataFrame
df = pd.DataFrame(results)
df.to_csv(f'{args.dataset_name}/predict_weights.csv', index=False)

convergence_threshold = 1e-4
for gene, cluster_losses in gene_cluster_losses.items():
    plt.figure(figsize=(6, 4))
    for cluster, losses in cluster_losses:
        max_xlim = len(losses)
        for i in range(len(losses) - 1):
            if abs(losses[i + 1] - losses[i]) < convergence_threshold:
                max_xlim = i
                break
        plt.plot(losses[:max_xlim + 10], label=f'{cluster}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Loss over Epochs for {gene} in All Clusters')
    plt.legend(ncol=2, title="Clusters")
    plt.xlim(0, max_xlim + 10)
    plt.savefig(args.result_path + f'loss_{gene}.png', dpi=300, bbox_inches='tight', pad_inches=0.1)
    plt.close()
plt.figure(figsize=(6, 4))
for gene, losses in mid_cell_losses:
    max_xlim = len(losses)
    for i in range(len(losses) - 1):
        if abs(losses[i + 1] - losses[i]) < convergence_threshold:
            max_xlim = i
            break
    plt.plot(losses[:max_xlim + 10], label=f'{gene}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over Epochs for All Genes in Middle Cell')
plt.legend(ncol=2, title="Genes")
plt.xlim(0, max_xlim + 10)
plt.savefig(args.result_path + 'loss_mid_cell.png', dpi=300, bbox_inches='tight', pad_inches=0.1)
plt.close()

# # ------- 新增：写入预测速度并绘制速度流图 -------

# # 1. 将预测速度写入adata.layers，每个基因一个layer
# import scanpy as sc

# for gene in adata.uns['selected_genes']:
#     layer_name = f'pred_velocity_{gene}'
#     # 初始化全零，形状(n_cells, n_genes)
#     pred_layer = np.zeros_like(adata.X if isinstance(adata.X, np.ndarray) else adata.X.toarray())
#     # 只给当前gene赋值为预测速度
#     gene_idx = adata.var_names.get_loc(gene)
#     pred_layer[:, gene_idx] = gene_pred_velocities[gene]
#     adata.layers[layer_name] = pred_layer

# # 2. 再写入一个合并所有选定基因的预测速度layer（仅这些基因有预测，其他为0）
# all_pred_velocity = np.zeros_like(adata.X if isinstance(adata.X, np.ndarray) else adata.X.toarray())
# for gene in adata.uns['selected_genes']:
#     gene_idx = adata.var_names.get_loc(gene)
#     all_pred_velocity[:, gene_idx] = gene_pred_velocities[gene]
# adata.layers['pred_velocity_selected_genes'] = all_pred_velocity

# # 3. 绘制每个基因的速度流图
# import scvelo as scv
# scv.settings.figdir = args.result_path + "velocity_stream/"
# os.makedirs(scv.settings.figdir, exist_ok=True)

# for gene in adata.uns['selected_genes']:
#     layer_name = f'pred_velocity_{gene}'
#     # scVelo的流图需要已计算embedding
#     scv.tl.velocity_graph(adata, n_jobs=args.n_jobs, vkey=layer_name)
#     scv.pl.velocity_embedding_stream(
#         adata, basis='umap', vkey=layer_name, color='clusters',
#         legend_loc='right margin', title=f'Predicted Velocity Stream for {gene}',
#         save=f'velocity_stream_{gene}.png', dpi=300
#     )

# # 4. 合并流图
# scv.tl.velocity_graph(adata, n_jobs=args.n_jobs, vkey='pred_velocity_selected_genes')
# scv.pl.velocity_embedding_stream(
#     adata, basis='umap', vkey='pred_velocity_selected_genes', color='clusters',
#     legend_loc='right margin', title='Predicted Velocity Stream for Selected Genes',
#     save='velocity_stream_selected_genes.png', dpi=300, n_neighbors=args.n_neighbors
# )

## 绘制调控网络图

In [ ]:
import os
import matplotlib.pyplot as plt
from matplotlib.patches import ArrowStyle
import networkx as nx
import numpy as np
import math
# df.to_csv(f'{args.dataset_name}/predict_weights.csv', index=False)
df = pd.read_csv(f'{args.dataset_name}/predict_weights.csv')
def normalize_weights(weights):
    min_val = np.min(weights)
    max_val = np.max(weights)
    if min_val == max_val:
        return [1.0 for _ in weights]
    normalized = [(0.5 + 2.5 * ((w - min_val) / (max_val - min_val))) for w in weights]
    return normalized

def adjust_node_sizes(expressions, scale_factor=1000, max_threshold=None):
    if len(expressions) == 0:
        return []
    raw_sizes = [scale_factor * expr for expr in expressions]
    if max_threshold is None:
        max_threshold = np.percentile(raw_sizes, 90)
    adjusted_sizes = [
        size if size <= max_threshold else max_threshold + np.sqrt(size - max_threshold)
        for size in raw_sizes
    ]
    return adjusted_sizes

def plot_regulatory_network_single(row, adata, dataset_name, outdir, cluster_name=None, cluster_idx=None):
    gene = row['Gene']
    cell = str(row['Cell'])
    weight_data = row
    if weight_data.empty:
        print(f"No weight data for cell: {cell}, gene: {gene}")
        return
    weight_data['Selected Genes'] = [item.strip("' ") for item in weight_data['Selected Genes'][1:-1].strip("' ").split(",")]
    weight_data['Selected Weights'] = [float(num) for num in weight_data['Selected Weights'].strip('[]').split()]
    # 选中目标基因及其权重
    selected_index = [i in adata.var_names for i in weight_data['Selected Genes']]
    selected_genes_all = [weight_data['Selected Genes'][i] for i, _ in enumerate(selected_index) if _ == True]
    selected_weights_all = [weight_data['Selected Weights'][i] for i, _ in enumerate(selected_index) if _ == True]
    if all(np.isnan(x) for x in selected_weights_all):
        print(f"No valid weights for cell: {cell}, gene: {gene}")
        return

    # ======== 新增：只取权重绝对值最大的前30个基因 ========
    # 得到排序后的索引
    num_max = 30
    idx_sort = np.argsort(np.abs(selected_weights_all))[::-1]  # 按绝对值降序
    idx_top = idx_sort[:num_max]
    selected_genes = [selected_genes_all[i] for i in idx_top]
    selected_weights = [selected_weights_all[i] for i in idx_top]
    # =====================================================

    # 获取表达量
    expressions = [
        adata.layers['Ms'][adata.obs_names.get_loc(cell), adata.var_names.get_loc(gene_name)]
        for gene_name in selected_genes
    ]
    key = gene
    G = nx.DiGraph()
    node_sizes = {}
    normalized_weights = normalize_weights(selected_weights)
    node_sizes_adjusted = adjust_node_sizes(expressions, scale_factor=1000)

    nodes_info = []
    for i in range(len(selected_weights)):
        source_node = selected_genes[i]
        weight = selected_weights[i]
        normalized_weight = normalized_weights[i]
        node_size = node_sizes_adjusted[i]
        nodes_info.append({
            'node': source_node,
            'weight': weight,
            'normalized_weight': normalized_weight,
            'size': node_size
        })
    nodes_info.sort(key=lambda x: x['size'], reverse=True)

    edges = []
    edge_widths = []
    is_positive = []
    for node_info in nodes_info:
        source_node = node_info['node']
        weight = node_info['weight']
        normalized_weight = node_info['normalized_weight']
        node_size = node_info['size']
        G.add_edge(source_node, key, weight=weight)
        node_sizes[source_node] = int(node_size)
        edges.append((source_node, key))
        edge_widths.append(normalized_weight)
        is_positive.append(weight > 0)

    pos = nx.circular_layout(G)
    pos[key] = (0, 0)
    node_sizes[key] = np.mean(list(node_sizes.values())) * 10 if node_sizes else 1000

    nodes = list(G.nodes())
    node_colors = ["yellow" for _ in nodes]
    node_colors[nodes.index(key)] = "lightgreen"
    sizes = [node_sizes[node] for node in nodes]

    fig, ax = plt.subplots(figsize=(4,4), dpi=300)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=sizes)
    for edge, width, is_pos in zip(edges, edge_widths, is_positive):
        color = "#3B4CC0"
        arrowstyle = "simple" if is_pos else ArrowStyle.BarAB(widthA=0, angleA=0, widthB=0.3, angleB=0)
        nx.draw_networkx_edges(
            G, pos, ax=ax, edgelist=[edge], edge_color=color, width=1,
            arrowstyle=arrowstyle, arrowsize=14, node_size=[size*1.2 for size in sizes]
        )
        nx.draw_networkx_edges(
            G, pos, ax=ax, edgelist=[edge], edge_color=color, width=width,
            arrowstyle="-", node_size=[size*1.28 for size in sizes]
        )

    for node, (x, y) in pos.items():
        if node != key:
            angle = np.arctan2(y, x) * 180 / np.pi
            alignment = 'center'
            if angle > 90 or angle < -90:
                angle += 180
            ax.text(
                x, y, node, fontsize=14, ha=alignment, va='center',
                rotation=angle, rotation_mode='anchor'
            )
        else:
            ax.text(x, y, node, fontsize=16, ha='center', va='center')
    # ax.set_title(f"Target Gene: {gene}")
    ax.set_aspect('equal')
    ax.axis('off')
    plt.tight_layout()

    # 文件名
    if cluster_name is not None and cluster_idx is not None:
        # 基因-簇网络
        filename = f"{cluster_idx:02d}_{dataset_name}_{gene}_{cluster_name}.svg"
        save_dir = os.path.join(outdir, "networks_by_gene", gene)
    else:
        # 中间细胞网络
        filename = f"{dataset_name}_mid_cell_{gene}.svg"
        save_dir = os.path.join(outdir, "networks_mid_cell")
    os.makedirs(save_dir, exist_ok=True)
    plt.savefig(os.path.join(save_dir, filename), dpi=300)
    plt.close()

# 获取伪时间排序簇
cluster2idx = {clu: i+1 for i, clu in enumerate(adata.obs['clusters_sorted'].cat.categories.tolist())}

# 非中间细胞
cluster_data = df[df['Cluster'] != 'MID']
# 中间细胞
mid_cell_data = df[df['Cluster'] == 'MID']

# 1. 每个基因在所有簇中单独画图
for gene, group in cluster_data.groupby('Gene'):
    for _, row in group.iterrows():
        cluster_name = row['Cluster']
        if cluster_name not in cluster2idx:
            print(f"Warning: Cluster {cluster_name} not in cluster_order, skip.")
            continue
        cluster_idx = cluster2idx[cluster_name]
        plot_regulatory_network_single(
            row, adata, args.dataset_name, args.result_path, cluster_name=cluster_name, cluster_idx=cluster_idx
        )

# 2. 所有基因在中间细胞上单独画图
for gene, group in mid_cell_data.groupby('Gene'):
    for _, row in group.iterrows():
        plot_regulatory_network_single(
            row, adata, args.dataset_name, args.result_path, cluster_name=None, cluster_idx=None
        )

# 新增：使用全部基因全部细胞寻找最优tau并预测速度

In [ ]:
import numpy as np
import pandas as pd
import scvelo as scv
import gc
from tqdm import tqdm
from collections import defaultdict
from joblib import Parallel, delayed

adata = sc.read(args.result_path + args.dataset_name + '_pp.h5ad')
print(adata.layers)
# adata = sc.read(r"E:\Administrator\Down\intestinal_organoid_pp_with_pred_velocity.h5ad")
# print(adata.layers)



In [ ]:
# scv.tl.velocity_graph(adata, vkey='pred_velocity', n_jobs=args.n_jobs)
# scv.pl.velocity_embedding_stream(
#     adata, basis='umap', vkey='pred_velocity', color='clusters',
#     legend_loc='right margin', title='Predicted by cong Velocity Stream',
#     save=args.result_path + 'cong_stream_predicted.png', dpi=300
# )

In [ ]:
# # scv.tl.velocity_graph(adata, vkey='pred_velocity_scv_dynamical', n_jobs=args.n_jobs)
# scv.pl.velocity_embedding_stream(
#     adata, basis='umap', vkey='pred_velocity_scv_dynamical', color='clusters',
#     legend_loc='right margin', title='Predicted by scVelo(dynamical) Velocity Stream',
#     save=args.result_path + 'scv_velocity_dynamical_stream_predicted.png', dpi=300
# )

In [ ]:
scv.pp.moments(adata, n_pcs=30, n_neighbors=args.n_neighbors)
# scv.tl.recover_dynamics(adata, n_jobs=args.n_jobs)
# velocity_file = 'v_cong.csv' if args.dataset_name == 'cong_velocity' else f'v_{args.dataset_name}.csv'
# v_intestinal = pd.read_csv(velocity_file, header=None)
# adata.layers['velocity'] = v_intestinal.values.T
# scv.tl.velocity_graph(adata, n_jobs=args.n_jobs)

scv.tl.recover_dynamics(adata, var_names='velocity_scv_genes', n_jobs=args.n_jobs)
scv.tl.velocity(adata, vkey='velocity_scv_dynamical', mode='dynamical')
scv.tl.velocity_graph(adata, vkey='velocity_scv_dynamical', n_jobs=args.n_jobs)

scv.tl.velocity(adata, vkey='velocity_scv_stochastic', mode='stochastic')
scv.tl.velocity_graph(adata, vkey='velocity_scv_stochastic', n_jobs=args.n_jobs)

scv.tl.velocity(adata, vkey='velocity_scv_deterministic', mode='deterministic')
scv.tl.velocity_graph(adata, vkey='velocity_scv_deterministic', n_jobs=args.n_jobs)

In [ ]:
scv.pl.velocity_embedding_stream(
    adata, basis='umap', vkey='velocity_scv_deterministic', color='clusters',
    legend_loc='right margin', title='scVelo(deterministic) Velocity Stream',
    save=args.result_path + args.dataset_name + '_scv_velocity_deterministic_stream.png', dpi=300
)

In [ ]:
scv.pl.velocity_embedding_stream(
    adata, basis='umap', vkey='velocity_scv_stochastic', color='clusters',
    legend_loc='right margin', title='scVelo(stochastic) Velocity Stream',
    save=args.result_path + args.dataset_name + '_scv_velocity_stochastic_stream.png', dpi=300
)

In [ ]:
scv.pl.velocity_embedding_stream(
    adata, basis='umap', vkey='velocity_scv_dynamical', color='clusters',
    legend_loc='right margin', title='scVelo(dynamics) Velocity Stream',
    save=args.result_path + args.dataset_name + '_scv_velocity_dynamics_stream.png', dpi=300
)

In [ ]:
# np.sum(np.any(adata.layers['pred_velocity'] != 0, axis=0))


In [ ]:
scv.pl.velocity_embedding_stream(
    adata, basis='umap', vkey='velocity', color='clusters',
    legend_loc='right margin', title='cong Velocity Stream',
    save=args.result_path + args.dataset_name + '_cong_velocity_stream.png', dpi=300
)